## Dataset download + image cache builder (Fakeddit)

This notebook:
- Loads the **multimodal** split (TSV) from Google Drive
- Filters to rows that actually have an image URL
- Downloads images into a local folder (cached by `id`)
- Verifies images are readable (drops/cleans failures)
- Writes a **clean CSV** that only contains rows with valid images

### Expected inputs
- A TSV like `multimodal_train.tsv` containing at least: `id`, `clean_title`, `image_url`, `hasImage`, `6_way_label`

### Outputs
- `IMAGEDIR/` containing `{id}.jpg`
- `CLEANDFPATH` CSV containing only rows with a usable image

### Notes
- The paths below are Colab/Drive paths. Update them if you run locally.
- Re-running is safe: already-downloaded images are reused from disk.


In [ ]:
# Core utilities
from urllib import request
import os

# Data
import pandas as pd
import numpy as np

# Image IO / validation
from PIL import Image

# Colab: mount Google Drive so we can read/write data under `/content/drive/...`
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv(
    '/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/multimodal_train.tsv',
    sep='\t'
)
df.drop(['2_way_label', '3_way_label', 'title'], axis = 1, inplace =True)
df.head(10)

## 1) Load the TSV and keep only relevant columns

We load the multimodal split, then drop columns not used in the image-download pipeline.


In [ ]:
IMAGEDIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images"
CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"


In [ ]:

from sklearn.model_selection import train_test_split

df, df_backup = train_test_split(
    df,
    test_size=0.95,
    shuffle=True,
    # To maintain percentage of samples per class as given by original dataset
    stratify=df["6_way_label"]
)
df.reset_index(drop=True, inplace=True)
df
print(df['clean_title'].isnull().sum())
print(df['id'].isnull().sum())
print(df['hasImage'].isnull().sum())

# Check for how many rows the column hasImage would be False
print(df['hasImage'].value_counts())
from matplotlib import pyplot as plt
df['6_way_label'].plot(kind='hist', bins=20, title='6_way_label')

from urllib import request

# Replace NaN values with empty strings
df = df.replace(np.nan, '', regex=True)
df.fillna('', inplace=True)

print("After split len(df):", len(df))
print("After split len(df_backup):", len(df_backup))
print("Sample ratio kept:", len(df) / (len(df) + len(df_backup)))


## 2) (Optional) Subsample for faster experimentation

The original dataset is large. This split keeps only a small stratified subset (by `6_way_label`) so you can test the pipeline quickly.


In [ ]:
import os
import time
from urllib import request
from PIL import Image
import numpy as np
import pandas as pd

# Create the image cache folder if it doesn't exist yet
os.makedirs(IMAGEDIR, exist_ok=True)

# Defensive cleanup: downstream checks assume strings, not NaNs
df = df.replace(np.nan, '', regex=True).fillna('')

# We'll rebuild the dataframe from only rows whose images are valid on disk
valid_rows = []

# Minimal progress stats so long runs are easier to monitor
stats = {
    "processed": 0,
    "eligible": 0,
    "cached": 0,
    "downloaded": 0,
    "verified": 0,
    "skipped_no_image": 0,
    "failed": 0
}

LOG_EVERY = 100
start_time = time.time()
total_rows = len(df)

print(f"Starting image preparation for {total_rows} rows...")
print(f"Image dir: {IMAGEDIR}")
print(f"Clean CSV path: {CLEANDFPATH}")

for idx, (_, row) in enumerate(df.iterrows(), start=1):
    stats["processed"] += 1

    # Skip rows that do not have an image (or have an empty/bad URL)
    if row["hasImage"] != True or row["image_url"] in ["", "nan"]:
        stats["skipped_no_image"] += 1
        if idx % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{idx}/{total_rows}] "
                f"cached={stats['cached']} downloaded={stats['downloaded']} "
                f"verified={stats['verified']} failed={stats['failed']} "
                f"skipped={stats['skipped_no_image']} elapsed={elapsed:.1f}s"
            )
        continue

    stats["eligible"] += 1

    img_id = str(row["id"])
    image_url = row["image_url"]
    path = os.path.join(IMAGEDIR, f"{img_id}.jpg")

    try:
        # Cache behavior: reuse if already present
        if not os.path.exists(path):
            with request.urlopen(image_url, timeout=20) as response:
                image_bytes = response.read()
            with open(path, "wb") as f:
                f.write(image_bytes)
            stats["downloaded"] += 1
            action = "downloaded"
        else:
            stats["cached"] += 1
            action = "cached"

        # Basic integrity check (raises on corruption)
        with Image.open(path) as img:
            img.verify()

        stats["verified"] += 1
        valid_rows.append(row.to_dict())

        if idx % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{idx}/{total_rows}] last={img_id} ({action}) "
                f"cached={stats['cached']} downloaded={stats['downloaded']} "
                f"verified={stats['verified']} failed={stats['failed']} "
                f"skipped={stats['skipped_no_image']} elapsed={elapsed:.1f}s"
            )

    except Exception as e:
        stats["failed"] += 1
        print(f"Failed for id={img_id}: {e}")

        # If a partial/corrupted file was written, remove it so the id can be retried
        if os.path.exists(path):
            os.remove(path)

# Persist a clean CSV with only rows whose images passed verification
df = pd.DataFrame(valid_rows).reset_index(drop=True)
df.to_csv(CLEANDFPATH, index=False)

elapsed = time.time() - start_time
print("\nImages ready. Clean df saved.")
print(f"Final rows kept: {len(df)}")
print(f"Stats: {stats}")
print(f"Total time: {elapsed:.1f}s")

## 3) Download + cache images (with verification)

For each row that has a usable `image_url`:
- Download `{id}.jpg` into `IMAGEDIR/` (or reuse if already present)
- Verify the file is a valid image (PIL `verify()`)
- Keep only rows that successfully verify

This produces a clean dataframe you can safely use for training.


In [ ]:
# Plotting images to test download
for i in range(5):
    path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + df["id"][i] + ".jpg"

    im= np.array(Image.open(path))

    print(im.shape)
    ax= plt.subplot(121)
    ax.imshow(im)

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load the RGBA image
image_path =  "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + df["id"][0] + ".jpg"
image = Image.open(image_path).convert("RGB")

# Split the image into individual channels
r, g, b = image.split()

# Plot each channel separately
plt.figure(figsize=(10, 5))

plt.subplot(1, 4, 1)
plt.imshow(r)
plt.title('Red Channel')

plt.subplot(1, 4, 2)
plt.imshow(g)
plt.title('Green Channel')

plt.subplot(1, 4, 3)
plt.imshow(b)
plt.title('Blue Channel')

#plt.subplot(1, 4, 4)
#plt.imshow(a)
#plt.title('Alpha Channel')

plt.tight_layout()
plt.show()

## 4) Post-download validation (optional)

If downloads were interrupted or some images are corrupted, these cells re-check the image folder and:
- drop rows with missing/corrupt images
- optionally delete bad files


In [ ]:
def validate_images(directory):
    corrupted_files = []

    # Walk through directory and sub-directories
    for index, row in df.iterrows():
      image_path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + row["id"] + ".jpg"
      try:
          with Image.open(image_path) as img:
              img.verify()
      except Exception as e:
          corrupted_files.append(image_path)
          print(f"Error with {image_path}: {e}")
          df.drop(index = index, axis = 0, inplace = True)

    return corrupted_files

# Example usage:
directory = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/"
corrupted_images = validate_images(directory)
if corrupted_images:
    print(f"Found {len(corrupted_images)} corrupted images.")
else:
    print("All images are valid!")
df.reset_index(drop=True, inplace=True)

In [ ]:
from PIL import Image
import os
import pandas as pd

def validate_images(frame, imagedir):
    valid_rows = []
    corrupted_files = []

    total_rows = len(frame)
    LOG_EVERY = 100

    print(f"Starting validation for {total_rows} rows...")
    print(f"Image dir: {imagedir}")
    print(f"Clean CSV path: {CLEANDFPATH}")

    for idx, (_, row) in enumerate(frame.iterrows(), start=1):
        img_id = str(row["id"])
        path = os.path.join(imagedir, f"{img_id}.jpg")

        try:
            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing file: {path}")

            with Image.open(path) as img:
                img.verify()

            valid_rows.append(row.to_dict())

            if idx % LOG_EVERY == 0:
                print(f"[{idx}/{total_rows}] validated last={img_id}")

        except Exception as e:
            corrupted_files.append(path)
            print(f"Error with {path}: {e}")

            if os.path.exists(path):
                os.remove(path)

    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    return valid_df, corrupted_files


df, corrupted_images = validate_images(df, IMAGEDIR)
df.to_csv(CLEANDFPATH, index=False)

if corrupted_images:
    print(f"Found {len(corrupted_images)} corrupted/missing images.")
else:
    print("All images are valid!")

print(f"Saved validated dataframe to {CLEANDFPATH}")
print(f"Rows after validation: {len(df)}")

In [ ]:
import os
import pandas as pd
from PIL import Image
from torchvision.transforms import v2

# Reload the latest validated CSV
df = pd.read_csv(CLEANDFPATH, sep=",")

new_size = (256, 256)
resize_transform = v2.Resize(new_size)

missing_ids = []
failed_ids = []
resized_count = 0

print(f"Starting resize for {len(df)} rows...")
print(f"Image dir: {IMAGEDIR}")
print(f"Clean CSV path: {CLEANDFPATH}")

for idx, row in df.iterrows():
    img_id = str(row["id"])
    image_path = os.path.join(IMAGEDIR, f"{img_id}.jpg")

    if not os.path.exists(image_path):
        missing_ids.append(img_id)
        print(f"Missing file for id={img_id}")
        continue

    try:
        image = Image.open(image_path).convert("RGB")
        resized_image = resize_transform(image)
        resized_image.save(image_path)
        resized_count += 1

        if (idx + 1) % 100 == 0:
            print(f"[{idx + 1}/{len(df)}] resized last={img_id}")

    except Exception as e:
        failed_ids.append(img_id)
        print(f"Resize failed for id={img_id}: {e}")


print(f"Resized successfully: {resized_count}")
print(f"Missing files: {len(missing_ids)}")
print(f"Failed resize: {len(failed_ids)}")
# Remove rows that still do not have usable image files


## 5) Normalize image sizes (resize)

Many vision backbones expect a consistent input size. This step resizes cached images in-place (and removes rows that still fail).


In [ ]:
bad_ids = set(missing_ids + failed_ids)
if bad_ids:
    df = df[~df["id"].astype(str).isin(bad_ids)].reset_index(drop=True)
    df.to_csv(CLEANDFPATH, index=False)

print("\nResize finished.")

print(f"Final rows kept: {len(df)}")
print(f"Updated clean df saved to: {CLEANDFPATH}")

In [ ]:
import pandas as pd
from PIL import Image
import os

df = pd.read_csv(CLEANDFPATH, sep=",")
print(df.shape)

sample_id = str(df.iloc[0]["id"])
sample_path = os.path.join(IMAGEDIR, f"{sample_id}.jpg")

with Image.open(sample_path) as img:
    print("Sample image size:", img.size)
    print("Sample image mode:", img.mode)

In [ ]:
import os
import pandas as pd
from datetime import datetime

print("CLEANDFPATH:", CLEANDFPATH)
print("Exists:", os.path.exists(CLEANDFPATH))
print("Modified:", datetime.fromtimestamp(os.path.getmtime(CLEANDFPATH)))

check_df = pd.read_csv(CLEANDFPATH, sep=",")
print("Shape on disk:", check_df.shape)
print(check_df.head(2))

In [ ]:
df.to_csv(CLEANDFPATH, index=False)
print(pd.read_csv(CLEANDFPATH, sep=",").shape)

In [ ]:
import shutil
import pandas as pd

CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"
FINAL_PATH  = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df_final.csv"

shutil.copy2(CLEANDFPATH, FINAL_PATH)

check = pd.read_csv(FINAL_PATH, sep=",")
print("Final shape:", check.shape)   # should print (17294, 13)
print("Columns:", check.columns.tolist())